# Week 8 Assignment - Agentic AI Pipeline (Single Agent System)

Name: Deeptesh Mohapatra

This project builds a small agentic AI pipeline: a single agent that reads a user query, decides
what to do, calls the right tool, checks the result, and returns a structured answer. It is built
as a stateful directed graph, which is the idea behind agent frameworks like LangGraph, but here
it is written from scratch in plain Python so every part is visible.

What the pipeline shows (these are the core ideas of single agent systems):

1. Tools with a JSON schema - each tool declares its name, what it does, and its inputs.
2. A stateful directed graph - a shared state flows through nodes (tasks) along edges (flow).
3. Conditional routing - the query is sent to the right tool based on its intent.
4. Loops for retries - if a step fails temporarily, the graph loops back and tries again.
5. Error handling - tools use try/except, and failures are logged, not crashed on.
6. Trajectory tracking and evaluation - the whole sequence of steps is recorded and checked.
7. Performance metrics - task completion rate and average number of steps.
8. One agent playing several roles - the nodes act like a coordinator, specialists, and a
   reviewer working one after another.

## Setup

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

import re, json, logging
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
log = logging.getLogger('agent')
print('Setup done')

C:\Users\lenovo\Desktop\Celebal_Internship\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup done


## 1. Tools and their JSON schema

Each tool is a plain Python function plus a JSON schema that describes it. The schema is a
standard structure (name, description, and the input parameters with their types) so the agent
knows what each tool expects. This is the same idea used by real tool-calling agents.

I define three tools:

- calculator: safely evaluates a maths expression.
- keyword_extractor: pulls the top keywords out of a piece of text.
- general_llm: answers a general question with a small language model (flan-t5).

In [2]:
# --- Tool 1: calculator ---
def calculator(expression):
    # only allow safe maths characters, then evaluate
    if not re.fullmatch(r'[0-9+\-*/(). %]+', expression.strip()):
        raise ValueError('expression contains characters that are not allowed')
    return eval(expression, {'__builtins__': {}}, {})

# --- Tool 2: keyword extractor ---
STOPWORDS = {'the','and','for','with','that','this','from','your','have','about','into','using'}
def keyword_extractor(text):
    words = re.findall(r'[a-zA-Z]+', text.lower())
    seen, keywords = set(), []
    for w in words:
        if len(w) > 4 and w not in STOPWORDS and w not in seen:
            seen.add(w); keywords.append(w)
    if not keywords:
        raise ValueError('no keywords found')
    return keywords[:5]

# --- Tool 3: general language-model answer ---
_tok = AutoTokenizer.from_pretrained('google/flan-t5-base')
_llm = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
def general_llm(query):
    ids = _tok(query, return_tensors='pt', truncation=True, max_length=256)
    with torch.no_grad():
        out = _llm.generate(**ids, max_new_tokens=60, num_beams=4, no_repeat_ngram_size=3)
    return _tok.decode(out[0], skip_special_tokens=True)

# JSON schema describing every tool
TOOL_SCHEMA = {
    'calculator': {
        'name': 'calculator',
        'description': 'Evaluate an arithmetic expression and return the number.',
        'parameters': {'type': 'object',
                       'properties': {'expression': {'type': 'string'}},
                       'required': ['expression']},
    },
    'keyword_extractor': {
        'name': 'keyword_extractor',
        'description': 'Return the top keywords found in a piece of text.',
        'parameters': {'type': 'object',
                       'properties': {'text': {'type': 'string'}},
                       'required': ['text']},
    },
    'general_llm': {
        'name': 'general_llm',
        'description': 'Answer a general question in natural language.',
        'parameters': {'type': 'object',
                       'properties': {'query': {'type': 'string'}},
                       'required': ['query']},
    },
}
print(json.dumps(TOOL_SCHEMA['calculator'], indent=2))

[INFO] HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


[WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/config.json "HTTP/1.1 200 OK"


[INFO] HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/tokenizer_config.json "HTTP/1.1 200 OK"


...

[INFO] HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/config.json "HTTP/1.1 200 OK"


[INFO] HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"


[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/generation_config.json "HTTP/1.1 200 OK"


{
  "name": "calculator",
  "description": "Evaluate an arithmetic expression and return the number.",
  "parameters": {
    "type": "object",
    "properties": {
      "expression": {
        "type": "string"
      }
    },
    "required": [
      "expression"
    ]
  }
}


## 2. A stateful directed graph engine

A shared state (a dictionary) flows through the graph. Each node is a function that reads the
state and returns an updated state. Edges say which node runs next. A conditional edge picks the
next node based on the state, which is how routing and loops work. Every node visited is appended
to the state's trajectory so we can see exactly what the agent did.

In [3]:
class AgentGraph:
    def __init__(self):
        self.nodes = {}       # name -> function(state) -> state
        self.edges = {}       # name -> next node name
        self.cond = {}        # name -> function(state) -> next node name
        self.entry = None

    def add_node(self, name, fn):        self.nodes[name] = fn
    def add_edge(self, src, dst):        self.edges[src] = dst
    def add_conditional(self, src, fn):  self.cond[src] = fn
    def set_entry(self, name):           self.entry = name

    def invoke(self, state, max_steps=15):
        node = self.entry
        while node != 'END' and state['steps'] < max_steps:
            state = self.nodes[node](state)
            state['trajectory'].append(node)
            state['steps'] += 1
            if node in self.cond:
                node = self.cond[node](state)
            elif node in self.edges:
                node = self.edges[node]
            else:
                node = 'END'
        return state

def new_state(query, simulate_transient=False):
    return {'query': query, 'intent': None, 'result': None, 'status': 'pending',
            'error': None, 'attempts': 0, 'retries': 0, 'max_retries': 2,
            'steps': 0, 'trajectory': [], 'simulate_transient': simulate_transient}

## 3. The nodes and how they are wired together

The nodes play different roles, so a single agent behaves like a small team:

- router (the coordinator): reads the query and decides the intent.
- calculator / keyword / general (the specialists): each handles one kind of task.
- validator (the reviewer): checks whether the step succeeded. If a tool hit a temporary error
  and we still have retries left, it loops back to try again; otherwise it ends.

Routing and the retry loop are both done with conditional edges.

In [4]:
# ---- role: coordinator ----
def router_node(state):
    q = state['query'].lower()
    if 'calculate' in q or re.search(r'[0-9]\s*[-+*/]', q):
        state['intent'] = 'calculator'
    elif 'keyword' in q or 'keywords' in q:
        state['intent'] = 'keyword'
    else:
        state['intent'] = 'general'
    log.info('router: intent = %s', state['intent'])
    return state

def route_from_router(state):
    return {'calculator': 'calculator', 'keyword': 'keyword', 'general': 'general'}[state['intent']]

# ---- role: specialists ----
def calculator_node(state):
    state['attempts'] += 1
    try:
        # simulate a temporary failure on the first attempt (to show the retry loop)
        if state['simulate_transient'] and state['attempts'] == 1:
            raise RuntimeError('temporary tool failure')
        # pull the arithmetic expression out of the query (ignore words around it)
        matches = re.findall(r'[0-9+\-*/(). %]+', state['query'])
        expr = max(matches, key=len).strip() if matches else state['query']
        state['result'] = {'type': 'calculation', 'result': calculator(expr)}
        state['status'] = 'ok'
    except Exception as e:
        state['status'] = 'error'; state['error'] = str(e)
        log.warning('calculator failed (attempt %d): %s', state['attempts'], e)
    return state

def keyword_node(state):
    state['attempts'] += 1
    try:
        text = re.sub(r'(?i)keywords?', '', state['query'])
        text = re.sub(r'(?i)extract|from', '', text).strip(' :')
        state['result'] = {'type': 'keywords', 'result': keyword_extractor(text)}
        state['status'] = 'ok'
    except Exception as e:
        state['status'] = 'error'; state['error'] = str(e)
        log.warning('keyword tool failed: %s', e)
    return state

def general_node(state):
    state['attempts'] += 1
    try:
        state['result'] = {'type': 'general', 'result': general_llm(state['query'])}
        state['status'] = 'ok'
    except Exception as e:
        state['status'] = 'error'; state['error'] = str(e)
        log.warning('general tool failed: %s', e)
    return state

# ---- role: reviewer ----
def validator_node(state):
    if state['status'] == 'error' and state['retries'] < state['max_retries']:
        state['retries'] += 1
        log.info('validator: retrying (%d/%d)', state['retries'], state['max_retries'])
    return state

def route_from_validator(state):
    if state['status'] == 'error' and state['retries'] <= state['max_retries'] \
            and state['attempts'] <= state['max_retries']:
        return state['intent'] if state['intent'] != 'keyword' else 'keyword'
    return 'END'

# build the graph
graph = AgentGraph()
graph.add_node('router', router_node)
graph.add_node('calculator', calculator_node)
graph.add_node('keyword', keyword_node)
graph.add_node('general', general_node)
graph.add_node('validator', validator_node)
graph.set_entry('router')
graph.add_conditional('router', route_from_router)
graph.add_edge('calculator', 'validator')
graph.add_edge('keyword', 'validator')
graph.add_edge('general', 'validator')
graph.add_conditional('validator', route_from_validator)
print('Graph built with nodes:', list(graph.nodes.keys()))

Graph built with nodes: ['router', 'calculator', 'keyword', 'general', 'validator']


## 4. Running the agent

A small helper runs a query through the graph and prints the structured JSON answer plus the
trajectory (the exact path the agent took through the graph).

In [5]:
def run_agent(query, simulate_transient=False):
    state = new_state(query, simulate_transient)
    state = graph.invoke(state)
    print('Query     :', query)
    print('Intent    :', state['intent'])
    print('Trajectory:', ' -> '.join(state['trajectory']), '-> END')
    print('Status    :', state['status'], '| attempts:', state['attempts'], '| steps:', state['steps'])
    print('Answer    :', json.dumps(state['result']) if state['result'] else state['error'])
    print('-' * 84)
    return state

_ = run_agent('Calculate 23 * (4 + 5)')
_ = run_agent('Extract keywords from: Agentic pipelines route queries to specialised tools')
_ = run_agent('What is an autonomous agent in artificial intelligence?')

[INFO] router: intent = calculator


[INFO] router: intent = keyword


[INFO] router: intent = general


Query     : Calculate 23 * (4 + 5)
Intent    : calculator
Trajectory: router -> calculator -> validator -> END
Status    : ok | attempts: 1 | steps: 3
Answer    : {"type": "calculation", "result": 207}
------------------------------------------------------------------------------------
Query     : Extract keywords from: Agentic pipelines route queries to specialised tools
Intent    : keyword
Trajectory: router -> keyword -> validator -> END
Status    : ok | attempts: 1 | steps: 3
Answer    : {"type": "keywords", "result": ["agentic", "pipelines", "route", "queries", "specialised"]}
------------------------------------------------------------------------------------


Query     : What is an autonomous agent in artificial intelligence?
Intent    : general
Trajectory: router -> general -> validator -> END
Status    : ok | attempts: 1 | steps: 3
Answer    : {"type": "general", "result": "robot"}
------------------------------------------------------------------------------------


### 4.1 Error handling and the retry loop

Here the calculator hits a temporary failure on its first attempt. The validator sees the error,
loops back, and the tool succeeds on the retry - so the trajectory shows the calculator node
appearing twice. This is why loops matter: they add resilience instead of failing immediately.

In [6]:
_ = run_agent('Calculate 100 / 4', simulate_transient=True)

# and a query that genuinely cannot be solved (division by zero) - it retries, then ends gracefully
_ = run_agent('Calculate 10 / (2 - 2)')

[INFO] router: intent = calculator


[WARNING] calculator failed (attempt 1): temporary tool failure


[INFO] validator: retrying (1/2)


[INFO] router: intent = calculator


[WARNING] calculator failed (attempt 1): division by zero


[INFO] validator: retrying (1/2)


[WARNING] calculator failed (attempt 2): division by zero


[INFO] validator: retrying (2/2)


[WARNING] calculator failed (attempt 3): division by zero


Query     : Calculate 100 / 4
Intent    : calculator
Trajectory: router -> calculator -> validator -> calculator -> validator -> END
Status    : ok | attempts: 2 | steps: 5
Answer    : {"type": "calculation", "result": 25.0}
------------------------------------------------------------------------------------
Query     : Calculate 10 / (2 - 2)
Intent    : calculator
Trajectory: router -> calculator -> validator -> calculator -> validator -> calculator -> validator -> END
Status    : error | attempts: 3 | steps: 7
Answer    : division by zero
------------------------------------------------------------------------------------


## 5. Trajectory evaluation and performance metrics

Evaluating an agent is not just about the final answer - it is also about whether it took the
right path. I run a small labelled test set and check two things:

- Routing accuracy (a trajectory check): did the agent send each query to the correct tool?
- Task completion rate: what share of queries returned a valid answer?

I also report the average number of steps, which is a simple cost metric.

In [7]:
test_set = [
    ('Calculate 12 + 8 * 2', 'calculator'),
    ('what is 45 - 9', 'calculator'),
    ('Extract keywords from: machine learning models need clean training data', 'keyword'),
    ('find the keywords in this sentence about neural networks', 'keyword'),
    ('What is reinforcement learning?', 'general'),
    ('Explain what a large language model is', 'general'),
]

correct_route, completed, total_steps = 0, 0, 0
for query, expected in test_set:
    st = new_state(query)
    st = graph.invoke(st)
    predicted = st['intent']
    routed_ok = (predicted == expected)
    correct_route += routed_ok
    completed += (st['status'] == 'ok')
    total_steps += st['steps']
    print(f"{'OK ' if routed_ok else 'BAD'} | expected {expected:10s} got {predicted:10s} | "
          f"status {st['status']:5s} | steps {st['steps']} | {query[:45]}")

n = len(test_set)
print('\n--- Metrics ---')
print('Routing accuracy    :', round(100 * correct_route / n, 1), '%')
print('Task completion rate:', round(100 * completed / n, 1), '%')
print('Average steps (cost):', round(total_steps / n, 2))

[INFO] router: intent = calculator


[INFO] router: intent = calculator


[INFO] router: intent = keyword


[INFO] router: intent = keyword


[INFO] router: intent = general


OK  | expected calculator got calculator | status ok    | steps 3 | Calculate 12 + 8 * 2
OK  | expected calculator got calculator | status ok    | steps 3 | what is 45 - 9
OK  | expected keyword    got keyword    | status ok    | steps 3 | Extract keywords from: machine learning model
OK  | expected keyword    got keyword    | status ok    | steps 3 | find the keywords in this sentence about neur


[INFO] router: intent = general


OK  | expected general    got general    | status ok    | steps 3 | What is reinforcement learning?


OK  | expected general    got general    | status ok    | steps 3 | Explain what a large language model is

--- Metrics ---
Routing accuracy    : 100.0 %
Task completion rate: 100.0 %
Average steps (cost): 3.0


## Conclusion

This project builds a working single agent pipeline as a stateful directed graph, from scratch:

1. Tools with JSON schemas - a calculator, a keyword extractor, and a language model, each
   declaring its name, purpose and inputs.
2. A graph engine where a shared state flows through nodes along edges, with the full trajectory
   recorded.
3. Conditional routing that sends each query to the right tool based on its intent.
4. A retry loop through the validator that recovers from a temporary failure instead of crashing.
5. Error handling with try/except and logging in every tool.
6. Trajectory evaluation and metrics - routing accuracy, task completion rate, and average steps.
7. One agent that behaves like a small team: a coordinator, three specialists, and a reviewer.

### What worked

- Writing the graph from scratch made every idea (nodes, edges, conditional routing, loops)
  concrete and easy to follow.
- Keyword based routing was simple and reliable for choosing the right tool.
- Recording the trajectory made both debugging and evaluation straightforward.

### Future enhancements

- Use a framework such as LangGraph to manage larger graphs and state.
- Let the language model itself choose the tool (function calling) for more flexible routing.
- Run independent tools in parallel, and add per-tool cost and latency tracking.
- Extend evaluation with an LLM-as-judge score on the quality of the final answers.